# Comparative Analysis: Adaptive MPC with Profile Likelihood Calculation 
This notebook runs the **unadapted MPC** and **adaptive MPC** simulation with **Profile Likelihood (PL)** calculations within the MHE horizon at each step. 
The initial parameter values are loaded from `Set_up_general.py` (via `initial_para`), which are defined in log10 space. 

It plots the Profile Likelihood objective function contours (heat plots) over time for each parameter. 


In [1]:
# add path 
from pathlib import Path 
import sys 

# Dynamically resolve project root containing solver_config.py 
project_root = Path.cwd().resolve() 
while project_root != project_root.parent and not (project_root / "solver_config.py").exists(): 
    project_root = project_root.parent 

if str(project_root) not in sys.path: 
    sys.path.insert(0, str(project_root)) 

# set up 
from Set_up_general import * # for all relevant other files - general setup + figures 
from MPC_MHE_PL.simulation_loop import run_simulation # simulation loop runner 


CasADi - 2026-10-19 10:04:28 WARNING("The options 't0', 'tf', 'grid' and 'output_t0' have been deprecated.
The same functionality is provided by providing additional input arguments to the 'integrator' function, in particular:
 * Call integrator(..., t0, tf, options) for a single output time, or
 * Call integrator(..., t0, grid, options) for multiple grid points.
The legacy 'output_t0' option can be emulated by including or excluding 't0' in 'grid'.
Backwards compatibility is provided in this release only.") [.../casadi/core/integrator.cpp:692]


run MPC benchmark (plant parameter)

In [2]:
# Run MPC benchmark simulation using true plant parameters (no MHE, no PL) 

Y_measurements, U_applied, Para_MHE, X_real_tk, Broken_mhe, Broken_mpc, Obj_tk, Para_PL, Broken_PL_tk, t_matrix, t_mean = run_simulation(
    initial_para_log=np.log10(para_plant), N_MPC=N_MPC, N_MHE=N_MHE, P=P, use_MHE=0, use_PL=0
)

results_MPC_benchmark = {
    'p_start_log': np.log10(para_plant),
    'Y_adapted': Y_measurements,
    'U_applied': U_applied,
    'X_real_tk': X_real_tk,
    'Broken_mpc': Broken_mpc,
    't_matrix': t_matrix,
    't_mean': t_mean,
}



******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

      solver  :   t_proc      (avg)   t_wall      (avg)    n_eval
       nlp_f  |        0 (       0) 318.00us (  6.12us)        52
       nlp_g  |   8.00ms (153.85us)   8.76ms (168.44us)        52
  nlp_grad_f  |   1.00ms ( 19.23us) 919.00us ( 17.67us)        52
  nlp_hess_l  |  54.00ms (  1.08ms)  54.93ms (  1.10ms)        50
   nlp_jac_g  |  26.00ms (500.00us)  28.38ms (545.79us)        52
       total  | 277.00ms (277.00ms) 276.82ms (276.82ms)         1
      solver  :   t_proc      (avg)   t_wall      (avg)    n_eval
       nlp_f  |        0 (       0) 330.00us (  7.02us)        47
       nlp_g  |   7

In [12]:
# Save benchmark results to dataset_MPC_benchmark.npz 
save_dict = {'N0': N0, 't': t}
# Save the initial parameter values (log10)
save_dict[f'p_start_log'] = results_MPC_benchmark['p_start_log']

# Save benchmark MPC results # 
save_dict[f'Y_adapted'] = results_MPC_benchmark['Y_adapted']
save_dict[f'U_applied'] = results_MPC_benchmark['U_applied']
save_dict[f'X_real_tk'] = results_MPC_benchmark['X_real_tk']
save_dict[f'Broken_mpc'] = results_MPC_benchmark['Broken_mpc']
save_dict[f't_matrix'] = results_MPC_benchmark['t_matrix']
save_dict[f't_mean'] = results_MPC_benchmark['t_mean']

np.savez("dataset_MPC_benchmark.npz", **save_dict)


run for different parameter values

In [4]:
# Run nominal MPC simulations across parameter sets (no MHE, no PL) 
results_MPC = {}
n_sets = initial_para.shape[1]

for i in range(n_sets):
    p_start_log = initial_para[:, i]
    print(f"\n" + "="*80)
    print(f"RUNNING SIMULATION SET {i+1}/{n_sets} | Initial parameters (log10): {p_start_log}")
    print("="*80)
    
    # Run MPC without adaptation 
    print("--> Running MPC...")
    Y_measurements, U_applied, Para_MHE, X_real_tk, Broken_mhe, Broken_mpc, Obj_tk, Para_PL, Broken_PL_tk, t_matrix, t_mean = run_simulation(
        initial_para_log=p_start_log, N_MPC=N_MPC, N_MHE=N_MHE, P=P, use_MHE=0, use_PL=0
    )
    
    results_MPC[i] = {
        'p_start_log': p_start_log,
        'Y_adapted': Y_measurements,
        'U_applied': U_applied,
        'X_real_tk': X_real_tk,
        'Broken_mpc': Broken_mpc,
        't_matrix': t_matrix,
        't_mean': t_mean,
    }



RUNNING SIMULATION SET 1/4 | Initial parameters (log10): [-2.04575749 -2.80392409 -2.95860731]
--> Running MPC...
      solver  :   t_proc      (avg)   t_wall      (avg)    n_eval
       nlp_f  |   4.00ms (  9.09us)   2.68ms (  6.10us)       440
       nlp_g  |  89.00ms (202.27us)  92.76ms (210.81us)       440
  nlp_grad_f  |   4.00ms ( 19.80us)   3.75ms ( 18.57us)       202
  nlp_hess_l  | 227.00ms (  1.14ms) 227.18ms (  1.14ms)       200
   nlp_jac_g  | 134.00ms (663.37us) 114.90ms (568.84us)       202
       total  |   1.24 s (  1.24 s)   1.24 s (  1.24 s)         1
      solver  :   t_proc      (avg)   t_wall      (avg)    n_eval
       nlp_f  |   2.00ms (  6.47us)   1.94ms (  6.27us)       309
       nlp_g  |  58.00ms (187.70us)  59.62ms (192.94us)       309
  nlp_grad_f  |   2.00ms (  9.90us)   3.86ms ( 19.11us)       202
  nlp_hess_l  | 218.00ms (  1.09ms) 219.93ms (  1.10ms)       200
   nlp_jac_g  | 105.00ms (519.80us) 107.28ms (531.10us)       202
       total  |   1.73 s ( 

In [14]:
# Save nominal MPC results to dataset_MPC.npz 
save_dict = {'N0': N0, 't': t}
for i in range(initial_para.shape[1]):
    # Save the initial parameter values (log10)
    save_dict[f'p_start_log_{i}'] = results_MPC[i]['p_start_log']
    
    # Save nominal MPC results for set i # 
    save_dict[f'Y_adapted_{i}'] = results_MPC[i]['Y_adapted']
    save_dict[f'U_applied_{i}'] = results_MPC[i]['U_applied']
    save_dict[f'X_real_tk_{i}'] = results_MPC[i]['X_real_tk']
    save_dict[f'Broken_mpc_{i}'] = results_MPC[i]['Broken_mpc']
    save_dict[f't_matrix_{i}'] = results_MPC[i]['t_matrix']
    save_dict[f't_mean_{i}'] = results_MPC[i]['t_mean']

np.savez("dataset_MPC.npz", **save_dict)


run MPC_MHE with PL calculation

In [ ]:
# Run simulation for each initial parameter set with Profile Likelihood enabled
results_MPC_MHE_PL = {}
n_sets = initial_para.shape[1]

for i in range(n_sets):
    p_start_log = initial_para[:, i]
    print(f"\n" + "="*80)
    print(f"RUNNING SIMULATION SET {i+1}/{n_sets} | Initial parameters (log10): {p_start_log}")
    print("="*80)
    
    # Run adaptive MPC with PL
    print("--> Running adaptive MPC with Profile Likelihood...") # 
    Y_measurements, U_applied, Para_MHE, X_real_tk, Broken_mhe, Broken_mpc, Obj_tk, Para_PL, Broken_PL_tk, t_matrix, t_mean = run_simulation(
        initial_para_log=p_start_log, N_MPC=N_MPC, N_MHE=N_MHE, P=P, use_MHE=1, use_PL=1
    )
    
    results_MPC_MHE_PL[i] = {
        'p_start_log': p_start_log,
        'Y_adapted': Y_measurements,
        'U_applied': U_applied,
        'Para_MHE': Para_MHE,
        'X_real_tk': X_real_tk,
        'Broken_mhe': Broken_mhe,
        'Broken_mpc': Broken_mpc,
        'Obj_tk': Obj_tk,
        'Para_PL': Para_PL,
        'Broken_PL_tk': Broken_PL_tk,
        't_matrix': t_matrix,
        't_mean': t_mean,
    }



RUNNING SIMULATION SET 1/4 | Initial parameters (log10): [-2.04575749 -2.80392409 -2.95860731]
--> Running adaptive MPC with Profile Likelihood...
      solver  :   t_proc      (avg)   t_wall      (avg)    n_eval
       nlp_f  |   9.00ms ( 20.45us)   2.68ms (  6.10us)       440
       nlp_g  |  74.00ms (168.18us)  75.26ms (171.04us)       440
  nlp_grad_f  |   7.00ms ( 34.65us)   3.72ms ( 18.40us)       202
  nlp_hess_l  | 233.00ms (  1.17ms) 225.70ms (  1.13ms)       200
   nlp_jac_g  | 107.00ms (529.70us) 115.97ms (574.09us)       202
       total  |   1.22 s (  1.22 s)   1.22 s (  1.22 s)         1
      solver  :   t_proc      (avg)   t_wall      (avg)    n_eval
       nlp_f  |   2.00ms (  5.41us)   2.31ms (  6.24us)       370
       nlp_g  |  64.00ms (172.97us)  61.69ms (166.72us)       370
  nlp_grad_f  |   7.00ms ( 34.65us)   3.73ms ( 18.45us)       202
  nlp_hess_l  | 225.00ms (  1.13ms) 221.63ms (  1.11ms)       200
   nlp_jac_g  | 114.00ms (564.36us) 113.12ms (560.00us)     

CasADi - 2026-10-19 14:50:12 WARNING("KeyboardInterruptException") [.../casadi/interfaces/ipopt/ipopt_nlp.cpp:132]


SystemError: <built-in function Function_call> returned a result with an exception set

In [9]:
# Save adaptive MPC with PL results to dataset_MPC_MHE_PL.npz 
save_dict = {'N0': N0, 't': t}
for i in range(initial_para.shape[1]):
    # Save the initial parameter values (log10)
    save_dict[f'p_start_log_{i}'] = results_MPC_MHE_PL[i]['p_start_log']
    
    # Save adaptive MPC results for set i 
    save_dict[f'Y_adapted_{i}'] = results_MPC_MHE_PL[i]['Y_adapted']
    save_dict[f'U_applied_{i}'] = results_MPC_MHE_PL[i]['U_applied']
    save_dict[f'Para_MHE_{i}'] = results_MPC_MHE_PL[i]['Para_MHE']
    save_dict[f'X_real_tk_{i}'] = results_MPC_MHE_PL[i]['X_real_tk']
    save_dict[f'Broken_mhe_{i}'] = results_MPC_MHE_PL[i]['Broken_mhe']
    save_dict[f'Broken_mpc_{i}'] = results_MPC_MHE_PL[i]['Broken_mpc']
    save_dict[f'Obj_tk_{i}'] = results_MPC_MHE_PL[i]['Obj_tk']
    save_dict[f'Para_PL_{i}'] = results_MPC_MHE_PL[i]['Para_PL']
    save_dict[f'Broken_PL_tk_{i}'] = results_MPC_MHE_PL[i]['Broken_PL_tk']
    save_dict[f't_matrix_{i}'] = results_MPC_MHE_PL[i]['t_matrix']
    save_dict[f't_mean_{i}'] = results_MPC_MHE_PL[i]['t_mean']

np.savez("dataset_MPC_MHE_PL.npz", **save_dict)
